In [ ]:
# Check GPU
!nvidia-smi

In [2]:
# Install libraries
!pip install -q numpy scipy scikit-learn matplotlib torch torchvision seaborn

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import euclidean, cosine as cosine_dist, cityblock
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import Dataset, DataLoader
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")
print("✓ Libraries imported!")

## Part 1: Understanding Similarity Metrics

### Generate Sample Vectors

In [ ]:
# Create sample vectors
v1 = np.array([1.0, 2.0, 3.0])
v2 = np.array([2.0, 4.0, 6.0])  # Same direction, 2x magnitude
v3 = np.array([3.0, 1.0, 2.0])  # Different direction
v4 = np.array([-1.0, -2.0, -3.0])  # Opposite direction

print("Sample vectors:")
print(f"v1 = {v1}")
print(f"v2 = {v2} (same direction as v1, 2× magnitude)")
print(f"v3 = {v3} (different direction)")
print(f"v4 = {v4} (opposite to v1)")

### Part 2: Euclidean Distance

**Formula**: $d(x, y) = \sqrt{\sum (x_i - y_i)^2}$

In [ ]:
# Manual calculation
def euclidean_manual(a, b):
    return np.sqrt(np.sum((a - b)**2))

# Compare vectors
print("Euclidean Distance:")
print(f"v1 ↔ v1: {euclidean_manual(v1, v1):.4f} (same vector)")
print(f"v1 ↔ v2: {euclidean_manual(v1, v2):.4f} (same direction, different magnitude)")
print(f"v1 ↔ v3: {euclidean_manual(v1, v3):.4f} (different direction)")
print(f"v1 ↔ v4: {euclidean_manual(v1, v4):.4f} (opposite direction)")

print("\nObservation: Sensitive to magnitude differences!")

### Part 3: Cosine Similarity

**Formula**: $cos(x, y) = \frac{x \cdot y}{||x|| \times ||y||}$

In [ ]:
# Manual calculation
def cosine_sim_manual(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine Similarity (range: -1 to 1):")
print(f"v1 ↔ v1: {cosine_sim_manual(v1, v1):.4f} (same vector)")
print(f"v1 ↔ v2: {cosine_sim_manual(v1, v2):.4f} (same direction!)")
print(f"v1 ↔ v3: {cosine_sim_manual(v1, v3):.4f} (different direction)")
print(f"v1 ↔ v4: {cosine_sim_manual(v1, v4):.4f} (opposite direction)")

print("\nObservation: Ignores magnitude, only cares about direction!")

### Part 4: Manhattan Distance (L1)

**Formula**: $d(x, y) = \sum |x_i - y_i|$

In [ ]:
# Manual calculation
def manhattan_manual(a, b):
    return np.sum(np.abs(a - b))

print("Manhattan Distance (L1):")
print(f"v1 ↔ v1: {manhattan_manual(v1, v1):.4f}")
print(f"v1 ↔ v2: {manhattan_manual(v1, v2):.4f}")
print(f"v1 ↔ v3: {manhattan_manual(v1, v3):.4f}")
print(f"v1 ↔ v4: {manhattan_manual(v1, v4):.4f}")

print("\nObservation: Sum of absolute differences (city block distance)")

### Part 5: Compare All Metrics

In [ ]:
# Generate random embeddings
np.random.seed(42)
embeddings = np.random.randn(50, 10)  # 50 samples, 10 dimensions

# Compute pairwise similarities/distances
euclidean_matrix = euclidean_distances(embeddings)
cosine_matrix = cosine_similarity(embeddings)
manhattan_matrix = manhattan_distances(embeddings)

print(f"Computed pairwise metrics for {embeddings.shape[0]} samples")
print(f"Matrix shapes: {euclidean_matrix.shape}")

In [ ]:
# Visualize similarity matrices
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Euclidean
sns.heatmap(euclidean_matrix[:20, :20], ax=axes[0], cmap='viridis', square=True)
axes[0].set_title('Euclidean Distance\n(Lower = More Similar)', fontsize=12)

# Cosine
sns.heatmap(cosine_matrix[:20, :20], ax=axes[1], cmap='RdYlGn', square=True,
            vmin=-1, vmax=1, center=0)
axes[1].set_title('Cosine Similarity\n(Higher = More Similar)', fontsize=12)

# Manhattan
sns.heatmap(manhattan_matrix[:20, :20], ax=axes[2], cmap='plasma', square=True)
axes[2].set_title('Manhattan Distance\n(Lower = More Similar)', fontsize=12)

plt.tight_layout()
plt.show()

### Part 6: When to Use Which Metric?

In [ ]:
# Test case: Same direction, different magnitudes
doc1 = np.array([1, 2, 3])  # Short document
doc2 = np.array([10, 20, 30])  # Long document, same topic

euc_dist = euclidean_manual(doc1, doc2)
cos_sim = cosine_sim_manual(doc1, doc2)

print("Use Case: Document Similarity")
print(f"\nDoc1 (short): {doc1}")
print(f"Doc2 (long, same topic): {doc2}")
print(f"\nEuclidean distance: {euc_dist:.2f} (considers them DIFFERENT due to length)")
print(f"Cosine similarity: {cos_sim:.4f} (considers them SAME topic!)")
print("\n✓ Use Cosine for text: ignores document length!")

In [ ]:
# Decision Guide
guide = """
╔═══════════════════════════════════════════════════════════════════╗
║                SIMILARITY METRIC SELECTION GUIDE                  ║
╠═══════════════════════════════════════════════════════════════════╣
║ Metric        │ When to Use                │ Example Use Case     ║
╠═══════════════════════════════════════════════════════════════════╣
║ Cosine        │ Direction matters,         │ • Text embeddings    ║
║               │ magnitude doesn't          │ • Word2Vec           ║
║               │                            │ • Normalized features║
╠═══════════════════════════════════════════════════════════════════╣
║ Euclidean     │ Both direction AND         │ • Image pixels       ║
║               │ magnitude matter           │ • Physical distances ║
║               │                            │ • Continuous features║
╠═══════════════════════════════════════════════════════════════════╣
║ Manhattan     │ Robust to outliers,        │ • Count data         ║
║               │ interpretable              │ • Sparse features    ║
║               │                            │ • Grid-based data    ║
╚═══════════════════════════════════════════════════════════════════╝
"""
print(guide)

## Part 7: Siamese Network (Learned Similarity)

**Idea**: Learn what "similar" means from data!

In [ ]:
# Load MNIST
transform = transforms.ToTensor()
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

print(f"MNIST loaded: {len(mnist_train)} training samples")

In [ ]:
# Create pairs dataset
class SiameseMNIST(Dataset):
    def __init__(self, mnist_dataset):
        self.mnist = mnist_dataset

    def __len__(self):
        return len(self.mnist)

    def __getitem__(self, idx):
        # Get anchor image
        img1, label1 = self.mnist[idx]

        # 50% chance: same class (positive pair)
        if np.random.rand() > 0.5:
            # Find another image with same label
            while True:
                idx2 = np.random.randint(len(self.mnist))
                img2, label2 = self.mnist[idx2]
                if label2 == label1:
                    break
            target = 1  # Similar
        else:
            # Find image with different label
            while True:
                idx2 = np.random.randint(len(self.mnist))
                img2, label2 = self.mnist[idx2]
                if label2 != label1:
                    break
            target = 0  # Dissimilar

        return img1, img2, torch.tensor(target, dtype=torch.float32)

# Create datasets
siamese_train = SiameseMNIST(mnist_train)
train_loader = DataLoader(siamese_train, batch_size=64, shuffle=True)

print("✓ Siamese pairs dataset created")

In [ ]:
# Define Siamese Network
class SiameseNetwork(nn.Module):
    def __init__(self):
        super(SiameseNetwork, self).__init__()
        # Shared encoder
        self.encoder = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28*28, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 32)
        )

    def forward(self, img1, img2):
        # Encode both images with SAME encoder
        emb1 = self.encoder(img1)
        emb2 = self.encoder(img2)
        return emb1, emb2

# Contrastive loss
class ContrastiveLoss(nn.Module):
    def __init__(self, margin=2.0):
        super(ContrastiveLoss, self).__init__()
        self.margin = margin

    def forward(self, emb1, emb2, label):
        distance = F.pairwise_distance(emb1, emb2)
        loss = (label * distance.pow(2) +
                (1 - label) * F.relu(self.margin - distance).pow(2))
        return loss.mean()

# Create model
model = SiameseNetwork().to(device)
criterion = ContrastiveLoss(margin=2.0)
optimizer = optim.Adam(model.parameters(), lr=0.001)

print("✓ Siamese Network created")
print("  Architecture: 784 → 128 → 64 → 32")
print("  Loss: Contrastive (margin=2.0)")

In [ ]:
# Train Siamese Network
num_epochs = 5
losses = []

print("Training Siamese Network...")
model.train()

for epoch in range(num_epochs):
    epoch_loss = 0
    for img1, img2, labels in train_loader:
        img1, img2, labels = img1.to(device), img2.to(device), labels.to(device)

        # Forward
        emb1, emb2 = model(img1, img2)
        loss = criterion(emb1, emb2, labels)

        # Backward
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / len(train_loader)
    losses.append(avg_loss)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

print("\n✓ Training complete!")

In [ ]:
# Test similarity
model.eval()

# Get test images
img1, label1 = mnist_test[0]  # First test image
img2, label2 = mnist_test[1]  # Second (same class?)
img3, label3 = mnist_test[100]  # Different class

with torch.no_grad():
    # Encode
    emb1 = model.encoder(img1.unsqueeze(0).to(device))
    emb2 = model.encoder(img2.unsqueeze(0).to(device))
    emb3 = model.encoder(img3.unsqueeze(0).to(device))

    # Compute distances
    dist_12 = F.pairwise_distance(emb1, emb2).item()
    dist_13 = F.pairwise_distance(emb1, emb3).item()

print(f"\nLearned Similarity:")
print(f"Image 1 (label {label1}) ↔ Image 2 (label {label2}): distance = {dist_12:.4f}")
print(f"Image 1 (label {label1}) ↔ Image 3 (label {label3}): distance = {dist_13:.4f}")
print(f"\nNetwork learned: same class = smaller distance!")